# Native sub-1B LFM selective-transfer pilot

This notebook runs the one-seed screening pilot for the 814M document VLM. It is separate from the legacy LoRA ablation notebook. A passing run is screening evidence only; promotion requires the sealed three-seed sweep.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

ROOT = next((p for p in (Path.cwd(), Path.cwd().parent) if (p / 'pyproject.toml').is_file()), None)
if ROOT is None:
    subprocess.run(['git', 'clone', 'https://github.com/SangbumChoi/OCR.git'], check=True)
    ROOT = Path('OCR').resolve()
subprocess.run(['git', '-C', str(ROOT), 'fetch', 'origin', 'claude/new-session-w79q0i'], check=True)
subprocess.run(['git', '-C', str(ROOT), 'checkout', 'claude/new-session-w79q0i'], check=True)
subprocess.run(['git', '-C', str(ROOT), 'merge', '--ff-only', 'origin/claude/new-session-w79q0i'], check=True)
os.chdir(ROOT)
if shutil.which('apt-get'):
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'libpango-1.0-0', 'libpangoft2-1.0-0', 'libharfbuzz0b', 'libfontconfig1', 'fonts-liberation', 'fonts-noto-core', 'fonts-noto-cjk'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[student,student-gpu,newvlms,synth,finetune]'], check=True)
print('repo:', ROOT)


In [ ]:
import os
import wandb

if not os.environ.get('WANDB_API_KEY'):
    try:
        from google.colab import userdata
        key = userdata.get('WANDB_API_KEY')
        if key:
            os.environ['WANDB_API_KEY'] = key
    except Exception:
        pass
wandb.login()
print('W&B target: https://wandb.ai/sbdc/docvlm-ablation')


In [ ]:
# Fast contract check. This allocates no student checkpoint and emits compact output.
subprocess.run([sys.executable, 'scripts/run_lfm_transfer_pilot_colab.py', '--dry-run', '--poll-seconds', '0.1'], check=True)


In [ ]:
# Full screening run. Resume is enabled, so rerunning this cell skips valid completed stages.
subprocess.run([sys.executable, 'scripts/run_lfm_transfer_pilot_colab.py'], check=True)


In [ ]:
import json
from pathlib import Path

summary_path = Path('outputs/sweeps/docvlm-lfm-language-transfer-pilot/sweep_run_summary.json')
summary = json.loads(summary_path.read_text())
print(json.dumps({
    'status': summary.get('status'),
    'variants': [{'run': row.get('run'), 'status': row.get('status')} for row in summary.get('variants', [])],
    'comparison': summary.get('comparison'),
    'promotion': summary.get('promotion'),
}, indent=2))
print('W&B: https://wandb.ai/sbdc/docvlm-ablation')
